In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

df = pd.read_csv(root / "data" / "posts_raw.csv")
print(df.shape)
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())


(303, 15)
post_id           0
date              0
post_type         0
topic             0
caption_length    0
hashtags_count    0
video_duration    0
posting_hour      0
likes             5
comments          0
shares            0
saves             0
reach             0
impressions       0
followers         0
dtype: int64
Duplicates: 3


In [7]:
# 1. Remove duplicate rows
df = df.drop_duplicates()

# 2. Convert date text into real dates
df["date"] = pd.to_datetime(df["date"])

# 3. Fill missing likes with the median likes of the same post type
df["likes"] = df["likes"].fillna(
    df.groupby("post_type")["likes"].transform("median")
).astype(int)

print("Rows:", len(df))
print("Missing values left:", df.isnull().sum().sum())


Rows: 300
Missing values left: 0


In [8]:
# Engagement rate (%)
df["engagement_rate"] = (
    (df["likes"] + df["comments"] + df["shares"] + df["saves"])
    / df["reach"] * 100
)

# Save rate and share rate (%)
df["save_rate"] = df["saves"] / df["reach"] * 100
df["share_rate"] = df["shares"] / df["reach"] * 100

# Reach as a ratio of followers
df["reach_ratio"] = df["reach"] / df["followers"]

# Time-based features
df["day_of_week"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.to_period("M").astype(str)
df["is_prime_time"] = df["posting_hour"].between(18, 21).astype(int)

df[["post_id", "post_type", "reach", "engagement_rate", "reach_ratio"]].head()

,post_id,post_type,reach,engagement_rate,reach_ratio
0,1000,Reel,34990,9.977136,3.481592
1,1001,Reel,15345,9.038775,1.523379
2,1002,Image,14122,9.311712,1.393939
3,1003,Carousel,14848,11.382004,1.460986
4,1004,Carousel,12001,12.007333,1.177030


In [9]:
# Top third of posts = High, middle = Medium, bottom = Low
df["performance_level"] = pd.qcut(
    df["reach_ratio"], q=3, labels=["Low", "Medium", "High"]
)

print(df["performance_level"].value_counts())

performance_level
Low       100
Medium    100
High      100
Name: count, dtype: int64


In [10]:
print(df.groupby("post_type")[["reach", "engagement_rate", "save_rate", "share_rate"]].mean().round(2))
print()
print(df.groupby("is_prime_time")["reach"].mean().round(0))


              reach  engagement_rate  save_rate  share_rate
post_type                                                  
Carousel   21984.73             9.76       2.27        1.20
Image      17975.15             8.72       1.23        1.22
Reel       41029.91             9.48       1.31        1.84

is_prime_time
0    26785.0
1    31548.0
Name: reach, dtype: float64


In [11]:
df.to_csv(root / "data" / "cleaned_posts.csv", index=False)
print("Saved!", df.shape)


Saved! (300, 23)
